In [0]:
%sql
-- SCRIPT: 6.5_relatorio_mensal_deputados
-- LOCAL: 3_gold/src/6_monitor_presenca/
-- OBJETIVO: Gerar relatório mensal (retornar percentil de cada deputado). Fonte usando a tabela do /3_gold/6_monitor_presenca/6.1_analise_absenteismo 
-- ENTREGA: Relatório automático mensal para cada deputado com percentile em relação à média

CREATE OR REPLACE TABLE gold_relatorio_mensal_performance AS
WITH base_agrupada AS (
    SELECT 
        id_deputado,
        nome_deputado,
        partido,
        '2026-05' AS mes_referencia,
        COUNT(*) AS total_oportunidades_voto,
        SUM(CASE WHEN status_presenca = 'PRESENTE' THEN 1 ELSE 0 END) AS total_presencas
    FROM gold_monitor_absenteismo
    GROUP BY id_deputado, nome_deputado, partido
),
calculo_score AS (
    SELECT 
        *,
        (CAST(total_presencas AS DOUBLE) / CAST(total_oportunidades_voto AS DOUBLE)) * 100 AS score_engajamento
    FROM base_agrupada
),
ranking_final AS (
    SELECT 
        *,
        AVG(score_engajamento) OVER() AS media_engajamento_camara,
        PERCENT_RANK() OVER(ORDER BY score_engajamento) * 100 AS percentil_performance
    FROM calculo_score
)
SELECT 
    id_deputado,
    nome_deputado,
    partido,
    mes_referencia,
    ROUND(score_engajamento, 2) AS score_engajamento,
    ROUND(media_engajamento_camara, 2) AS media_engajamento_camara,
    ROUND(percentil_performance, 2) AS percentil_performance
FROM ranking_final;

SELECT * FROM gold_relatorio_mensal_performance ORDER BY percentil_performance DESC;